# 🕵️ Dia 20 — O ciclo OSINT, parte 2: coleta passiva automatizada com SpiderFoot

**Investigador Cyber 90 Dias** · Fase 2 · Rank Jutsu `C`

> Baseado no *Day 20* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-2-osint-digital-footprint/day-20.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

O SpiderFoot terminou: **485 dados** sobre `serraazul-pix.example`. Rafa circula uma linha do Summary: *Co-Hosted Site: 412*. "Você vai dizer para a Marta que o Pescador tem 412 sites? Me diz quantos você joga fora, por quê, e me prova que nenhum módulo bateu na porta do 203.0.113.77."

Neste notebook você vai:

1. **Ler a exportação CSV** do SpiderFoot e refazer a aba Summary.
2. **Jogar "ruído ou pista?"** duas vezes: com a pergunta do laboratório original e com a pergunta do golpe.
3. **Filtrar, gerar a tabela de falsos positivos e escrever um `.gexf`** para o Gephi.
4. **Varrer o Log** atrás de contato direto com o alvo.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Tudo o que é do caso é **fictício** (domínios `.example`, IPs de documentação).

## 1. A exportação do SpiderFoot

O CSV do SpiderFoot tem seis colunas: `Updated`, `Type`, `Module`, `Source`, `F/P`, `Data`. **`Source` é o dado que gerou este** (é isso que vira aresta no grafo) e `Data` é o dado novo (vira nó).

A célula recria a exportação sintética do caso, idêntica a [`caso-continuo/dados/dia-20/sf-isca-passive-01.csv`](https://github.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/tree/main/caso-continuo/dados/dia-20). Se você tiver um CSV de verdade da sua varredura do laboratório, troque `ARQUIVO`.

In [ ]:
import csv
from pathlib import Path

# Gerador determinístico dos dados do caso (mesmo código que gerou o CSV do repo).
def gerar_linhas():
    """Devolve as linhas (Updated, Type, Module, Source, F/P, Data) de uma
    exportação CSV sintética do SpiderFoot, caso de uso Passive,
    semente serraazul-pix.example. Determinística: sem aleatoriedade."""
    t = "2026-10-06 10:{:02d}:{:02d}"          # carimbos sintéticos (hora do SpiderFoot)
    L = []
    def add(i, tipo, mod, fonte, dado):
        L.append((t.format(10 + i // 60 % 30, i % 60), tipo, mod, fonte, "0", dado))
    s = "serraazul-pix.example"
    n = 0
    # Nomes e domínios do próprio site falso
    for nome in ["serraazul-pix.example", "www.serraazul-pix.example",
                 "admin.serraazul-pix.example", "cdn.serraazul-pix.example"]:
        add(n, "Internet Name", "sfp_crt", s, nome); n += 1
    add(n, "Domain Name", "sfp_dnsresolve", s, s); n += 1
    # Resolução e rede
    for nome in ["serraazul-pix.example", "www.serraazul-pix.example"]:
        add(n, "IP Address", "sfp_dnsresolve", nome, "203.0.113.77"); n += 1
    add(n, "IP Address", "sfp_dnsresolve", "cdn.serraazul-pix.example", "192.0.2.80"); n += 1
    add(n, "BGP AS Ownership", "sfp_bgpview", "203.0.113.77", "AS64501 HOSTBARATO-EXEMPLO"); n += 1
    add(n, "BGP AS Ownership", "sfp_bgpview", "192.0.2.80", "AS64500 CDN-EXEMPLO"); n += 1
    add(n, "Name Server (DNS NS Records)", "sfp_dnsresolve", s, "ns1.dnsbarato.example"); n += 1
    add(n, "Name Server (DNS NS Records)", "sfp_dnsresolve", s, "ns2.dnsbarato.example"); n += 1
    # Certificados (o do site falso foi emitido em 04/out, véspera do disparo)
    add(n, "SSL Certificate - Issued to", "sfp_crt", s, "serraazul-pix.example (emitido 2026-10-04)"); n += 1
    add(n, "SSL Certificate - Issued to", "sfp_crt", s, "admin.serraazul-pix.example (emitido 2026-10-04)"); n += 1
    # O identificador pequeno e específico
    add(n, "Web Analytics ID", "sfp_webanalytics", s, "G-ISCA7K2Q9P"); n += 1
    # E-mail do registro: serviço de privacidade (não identifica ninguém)
    add(n, "Email Address", "sfp_whois", s, "privacidade@proxy-registro.example"); n += 1
    # Domínios parecidos: dois imitam a cooperativa, três são coincidência
    for dom in ["serraazul-seguranca.example", "coopserra-azul.example",
                "serra-azul-tintas.example", "azulserra-imoveis.example", "serraazull.example"]:
        add(n, "Similar Domain", "sfp_similar", s, dom); n += 1
    for i in range(52):                            # mais 52 parecidos genéricos (ruído)
        add(n, "Similar Domain", "sfp_similar", s, f"serra{i:02d}-azul-loja.example"); n += 1
    # 412 sites co-hospedados: a hospedagem barata, não o golpe
    for i in range(412):
        add(n, "Co-Hosted Site", "sfp_robtex", "203.0.113.77", f"cliente{i:03d}-hostbarato.example"); n += 1
    return L

ARQUIVO = "sf-isca-passive-01.csv"     # troque pelo caminho do SEU export, se tiver um
if not Path(ARQUIVO).exists():
    with open(ARQUIVO, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["Updated", "Type", "Module", "Source", "F/P", "Data"])  # cabeçalho do SpiderFoot
        w.writerows(gerar_linhas())

with open(ARQUIVO, newline="", encoding="utf-8") as f:
    LINHAS = list(csv.DictReader(f))   # cada linha vira um dicionário {coluna: valor}
print(len(LINHAS), "dados lidos de", ARQUIVO)
print("Exemplo:", LINHAS[0])

In [ ]:
from collections import Counter

# Refaz a aba Summary: quantos dados de cada tipo.
resumo = Counter(l["Type"] for l in LINHAS)
for tipo, qtd in resumo.most_common():
    print(f"{tipo:<32}{qtd:>6}")

# A pergunta do Rafa: a que os 412 co-hospedados estão presos?
presos = Counter(l["Source"] for l in LINHAS if l["Type"] == "Co-Hosted Site")
print("\nCo-Hosted Site presos a:", dict(presos))

## 2. 🎮 Ruído ou pista?

Mesmos tipos de dado, **duas perguntas diferentes**:

- **Rodada A** — pergunta do laboratório original: *quais propriedades web a organização opera?*
- **Rodada B** — pergunta do caso: *quais domínios e hosts estão ligados à campanha que imita a Serra Azul?*

Responda **1** (ruído, sai do grafo) ou **2** (pista, fica). Repare no tipo que muda de resposta.

In [ ]:
# (tipo, resposta na rodada A, resposta na rodada B, explicação)
TIPOS = [
    ("Co-Hosted Site", "R", "R", "412 presos a um IP de hospedagem barata: vizinhos, não donos."),
    ("Similar Domain", "R", "P", "Na pegada da organização é coincidência; num golpe de imitação é o alvo da caça."),
    ("Web Analytics ID", "P", "P", "Pequeno e específico. Candidato a ponte (Dia 22)."),
    ("SSL Certificate - Issued to", "P", "P", "Datas de emissão entram na linha do tempo (emitido na véspera do disparo)."),
    ("Name Server (DNS NS Records)", "P", "P", "Fica no grafo, mas cuidado: DNS barato também é hub (Dia 22)."),
]
MODO_JOGO = True

for rodada, idx, pergunta in (("A", 1, "propriedades da organização"), ("B", 2, "campanha que imita a Serra Azul")):
    print(f"\n===== Rodada {rodada}: pergunta = {pergunta} =====")
    acertos = 0
    for tipo, *respostas, porque in TIPOS:
        certo = respostas[idx - 1]                     # resposta esperada nesta rodada
        if MODO_JOGO:
            r = ""
            while r not in ("1", "2"):                 # insiste até vir 1 ou 2
                r = input(f"  {tipo}: 1 = ruído, 2 = pista: ").strip()
            r = "R" if r == "1" else "P"               # traduz o número para a letra do gabarito
            acertos += (r == certo)
            print("   ✅" if r == certo else "   ❌", end=" ")
        print(f"  {tipo} → {'pista' if certo == 'P' else 'ruído'}. {porque}")
    if MODO_JOGO:
        print(f"Rodada {rodada}: {acertos}/{len(TIPOS)}")

print("\n🧠 O ruído é definido pela PERGUNTA, não pela ferramenta.")

## 3. Filtro com motivo, tabela de falsos positivos e GEXF

Dentro de **Similar Domain** ainda há uma triagem: o que imita **a cooperativa** é pista; o que só tem "serra" ou "azul" é coincidência. O critério precisa estar escrito. Abaixo ele é: *o nome contém "serraazul" ou "coopserra" (com ou sem hífen) e não contém palavras de outro ramo (loja, tintas, imoveis)*. Imperfeito? Sim. **Escrito e auditável?** Também. Ajuste e rode de novo.

In [ ]:
import re

def similar_e_pista(dominio):
    """Critério escrito para Similar Domain no caso Isca."""
    nome = dominio.lower().replace("-", "")                    # serra-azul → serraazul
    imita = ("serraazul" in nome) or ("coopserra" in nome)    # imita a cooperativa?
    outro_ramo = re.search(r"loja|tintas|imoveis", nome)      # sinais de outro negócio
    return imita and not outro_ramo

PERGUNTA = "Quais domínios e hosts estão ligados à campanha que imita a Serra Azul?"
mantidos, removidos = [], Counter()
motivos = {"Co-Hosted Site": "Presos a 203.0.113.77 (hospedagem compartilhada barata); vizinhança, não controle",
           "Similar Domain": "Parecido só com 'serra'/'azul'; outro ramo de negócio; sem relação com a cooperativa"}
pistas_similar = []

for l in LINHAS:
    if l["Type"] == "Co-Hosted Site":
        removidos["Co-Hosted Site"] += 1
    elif l["Type"] == "Similar Domain" and not similar_e_pista(l["Data"]):
        removidos["Similar Domain"] += 1
    else:
        mantidos.append(l)
        if l["Type"] == "Similar Domain":
            pistas_similar.append(l["Data"])

print(f"Pergunta: {PERGUNTA}\n")
print("| Nó ou grupo removido | Quantidade | Motivo |\n|---|---|---|")
for tipo, qtd in removidos.items():
    print(f"| {tipo} | {qtd} | {motivos[tipo]} |")
print(f"| Similar Domain (mantido de propósito) | {len(pistas_similar)} | Imitam a cooperativa; fila de confirmação do Dia 22: {', '.join(pistas_similar)} |")
print(f"\nNós que ficam: {len(mantidos)} dados (meta do original: até ~150 nós)")

In [ ]:
import xml.etree.ElementTree as ET

# Monta o grafo: cada valor (Source ou Data) é um nó; cada linha é uma aresta Source → Data.
ids = {}                                            # rótulo → id do nó
def no(rotulo):
    if rotulo not in ids:
        ids[rotulo] = str(len(ids))                 # ids sequenciais: "0", "1", ...
    return ids[rotulo]

tipo_do_no = {}
arestas = []
for l in mantidos:
    origem, destino = no(l["Source"]), no(l["Data"])
    tipo_do_no[l["Data"]] = l["Type"]              # o tipo vira atributo (cor no Gephi)
    if origem != destino:                           # evita laço da semente nela mesma
        arestas.append((origem, destino, l["Module"]))

gexf = ET.Element("gexf", xmlns="http://gexf.net/1.3", version="1.3")
grafo = ET.SubElement(gexf, "graph", defaultedgetype="undirected")
attrs = ET.SubElement(grafo, "attributes", **{"class": "node"})
ET.SubElement(attrs, "attribute", id="tipo", title="tipo", type="string")
nos_xml = ET.SubElement(grafo, "nodes")
for rotulo, i in ids.items():
    n = ET.SubElement(nos_xml, "node", id=i, label=rotulo)
    av = ET.SubElement(ET.SubElement(n, "attvalues"), "attvalue",
                       **{"for": "tipo", "value": tipo_do_no.get(rotulo, "Semente")})
arestas_xml = ET.SubElement(grafo, "edges")
for k, (a, b, modulo) in enumerate(arestas):
    ET.SubElement(arestas_xml, "edge", id=str(k), source=a, target=b, label=modulo)

ET.ElementTree(gexf).write("isca-filtrado.gexf", encoding="utf-8", xml_declaration=True)
ET.parse("isca-filtrado.gexf")                      # relê: se o XML estivesse quebrado, daria erro aqui
print(f"💾 isca-filtrado.gexf: {len(ids)} nós, {len(arestas)} arestas. Abra no Gephi (File > Open).")

## 4. 🔎 O Log denuncia

O Passo 9 do original: procurar no Log qualquer módulo que tenha feito requisição direta ao alvo. Consultar **terceiros** sobre o alvo (crt.sh, RDAP, API de BGP) é passivo. Buscar **no próprio** servidor do alvo não é.

In [ ]:
LOG = """2026-10-06 10:10:00 INFO  sfp__stor_db: Scan P2-ISCA-passive-01 started, use case Passive
2026-10-06 10:10:01 INFO  sfp_dnsresolve: Resolving serraazul-pix.example via public resolver 9.9.9.9
2026-10-06 10:10:03 INFO  sfp_crt: Querying crt.sh for %.serraazul-pix.example
2026-10-06 10:10:07 INFO  sfp_robtex: Querying third-party API for co-hosted sites on 203.0.113.77
2026-10-06 10:10:09 INFO  sfp_similar: Generating look-alike candidates for serraazul-pix.example
2026-10-06 10:10:12 INFO  sfp_whois: Querying RDAP for serraazul-pix.example
2026-10-06 10:10:15 INFO  sfp_spider: Fetching https://serraazul-pix.example/ (module enabled manually after use-case selection)
2026-10-06 10:10:16 INFO  sfp_webanalytics: Parsing web content from https://serraazul-pix.example/
2026-10-06 10:10:20 INFO  sfp_bgpview: Querying third-party API for 203.0.113.77
2026-10-06 10:40:02 INFO  sfp__stor_db: Scan P2-ISCA-passive-01 finished, 485 elements
"""

ALVOS = ("https://serraazul-pix.example", "http://serraazul-pix.example", "203.0.113.77")
for linha in LOG.splitlines():
    terceiro = "third-party" in linha or "crt.sh" in linha or "RDAP" in linha or "public resolver" in linha
    toca_alvo = any(a in linha for a in ALVOS) and ("Fetching" in linha or "Connecting" in linha)
    marca = "🚨 CONTATO DIRETO" if toca_alvo else ("   terceiro" if terceiro else "")
    print(f"{marca:<18} {linha}")

**O que anotar:** o `sfp_spider` foi ativado à mão depois da escolha do caso de uso, e buscou a página inicial do site falso. A linha seguinte (`sfp_webanalytics`) mostra que o **Web Analytics ID veio desse conteúdo**. Então:

1. registre no recon log que a varredura **não** ficou passiva, qual módulo e a que horas (`10:10:15`);
2. rode de novo sem o módulo;
3. **não use** o `G-ISCA7K2Q9P` como achado a partir desta varredura: confirme-o na sua captura do Dia 19 (leitura da página pública como visitante, com hash), que é a fonte limpa.

### 🧠 Camada 8

485 dados *parecem* uma descoberta. O trabalho do analista é medido pelo que ele **descarta com motivo**. E a regra decorada ("Similar Domain é ruído") é tão perigosa quanto não ter regra: isso é fixação funcional.

## ✅ Antes de fechar o Dia 20

- [ ] Summary refeito e os 412 presos a um IP só
- [ ] Rodadas A e B jogadas; você sabe dizer qual tipo mudou de resposta e por quê
- [ ] Tabela de falsos positivos com linha "mantido de propósito"
- [ ] `isca-filtrado.gexf` gerado (e aberto no Gephi, se tiver)
- [ ] Linha de contato direto achada e registrada

**Amanhã (Dia 21):** Maltego, um salto de cada vez. Do domínio ao AS, e a descoberta de que o `cdn.` do golpe não está onde o resto está.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>

In [ ]:
try:
    from google.colab import files
    files.download("isca-filtrado.gexf")
except ImportError:
    print("(Fora do Colab: o arquivo ficou em isca-filtrado.gexf)")